In [1]:
import { RecursiveSet as Set, Tuple, Value, flatMap } from 'recursive-set';
type Variable = string;
type Literal  = Variable | Tuple<['¬', Variable]>;
type Clause   = Set<Literal>;
type Clauses  = Set<Clause>;

In [2]:
function set<T extends Value>(...elements: T[]): Set<T> {
    return new Set(...elements);
}

In [3]:
function tpl<T extends Value[]>(...elements: T): Tuple<T> {
    return new Tuple(...elements);
}

In [4]:
function range(start: number, stop: number): Set<number> {
  return set(...Array.from(
      { length: stop - start + 1 }, 
      (_, index) => start + index));
}

## Sudoku

In [5]:
import { solve, Variable, Literal } from './07-Davis-Putnam-JW';

In [6]:
function neg(p: Variable): Literal {
    return tpl('¬', p);
}

The Finnish mathematician Arto Inkala claims to have created the [hardest sudoku](https://abcnews.go.com/blogs/headlines/2012/06/can-you-solve-the-hardest-ever-sudoku) ever.  It is shown below.
<img src="sudoku.png" alt="A game of Sudoku" width="50%">

The function `createSudoku` returns the Sudoku as a list of list.  This list contains `'*'` characters for those cells that are left unspecified.

In [7]:
function createPuzzle(): (number | '*')[][] {
    return [[ 8 , '*', '*', '*', '*', '*', '*', '*', '*'],
            ['*', '*',  3,   6 , '*', '*', '*', '*', '*'],
            ['*',  7 , '*', '*',  9 , '*',  2 , '*', '*'],
            ['*',  5 , '*', '*', '*',  7 , '*', '*', '*'],
            ['*', '*', '*', '*',  4 ,  5 ,  7 , '*', '*'],
            ['*', '*', '*',  1 , '*', '*', '*',  3 , '*'],
            ['*', '*',  1 , '*', '*', '*', '*',  6 ,  8 ],
            ['*', '*',  8 ,  5 , '*', '*', '*',  1 , '*'],
            ['*',  9 , '*', '*', '*', '*',  4 , '*', '*']
    ];
}

We will solve this Sudoku using the Davis-Putnam algorithm.  We use the following variables:
* `Q<r,c,d>` is a Boolean variable stating that the field in row `r` and column `c` holds the digit `d`.
  Here, `r`, `c`, `d` are all elements from the set $\{1,\cdots,9\}$.
    
The function `varName(row, col, digit)` returns a formated string that is interpreted as a variable name.

In [8]:
function varName(row: number, col: number, digit: number): string {
    return `Q<${row},${col},${digit}>`;
}

In [9]:
varName(1,2,3);

Q<1,2,3>


The function `atMostOne(S)` takes a set `S` of propositional variables as its argument.  It returns a set of clauses
expressing the fact that at most one of the variables of `S` is true.
$$ \texttt{atMostOne}(S) = \bigl\{ \{\neg p, \neg q\} \bigm| \langle p, q\rangle \in S \times S \wedge p \not= q\bigr\}. $$

In [10]:
function atMostOne(S: Set<Variable>): Set<Clause> {
    return S.cartesianProduct(S)
            .filterMap(([p, q]) => p != q,
                       ([p, q]) => set(neg(p), neg(q)));
}

The function `atLeastOne(S)` takes a set `S` of propositional variables as its argument.  It returns a set of clauses
expressing the fact that at least one of the variables of `S` is true.

In [11]:
function atLeastOne(S: Set<Variable>): Set<Clause> {
    return set(S);
}

The function `exactlyOne(S)` takes a set `S` of propositional variables as its argument.  It returns a set of clauses
expressing the fact that exactly one of the variables of `S` is true.

In [12]:
function exactlyOne(S: Set<Variable>): Set<Clause> {
    return atMostOne(S).union(atLeastOne(S));
}

In [13]:
exactlyOne(set('a', 'b', 'c'));

{{(¬, a), (¬, b)}, {(¬, a), (¬, c)}, {(¬, b), (¬, c)}, {a, b, c}}


A `Position` is a pair of coordinates of the form `(row, col)` where
`row` specifies the row and `col`specifies the column

In [14]:
type Position = Tuple<[number, number]>;

The function `exactlyOneForDigit` takes two arguments:
* An array `L` of pairs of indices representing Sudoku coordinates   as its first argument.
  The elements of `L` are pairs of the form `(row, col)`, where both `row` and `col` are elements of the set $\{1, \cdots, 9\}$.
* A specific `digit` as its second arguments.  
  `digit` is an element from the set `{1,...,9}`.

It returns a set of clauses expressing the fact that the given `digit` 
appears exactly once among the cells specified by the coordinate pairs in `L`.

In [15]:
function exactlyOneForDigit(L: Set<Position>, digit: number): Set<Clause> {
    return exactlyOne(L.map(([row, col]) => varName(row, col, digit)));
}

The function `exactlyOneForEveryDigit` takes an array `L` of pairs of indices as its argument.  The elements of `L` are pairs of the form
`(row, col)`, where both `row` and `col` are elements of the set $\{1, \cdots, 9\}$.
It returns a set of formulas expressing that all of the nine digits occurs at exactly one position in the list `L`.

In [16]:
function exactlyOneForEveryDigit(L: Set<Position>): Set<Clause> {
    return flatMap(range(1, 9), d => exactlyOneForDigit(L, d));
}

The function `exactlyOneDigit(row, col)` takes integers `row` and `col` as arguments.  These specify the row and column of a field in a Sudoku.  The function returns a set of clauses specifying that exactly one of the variables

* `Q<row,col,1>`, `Q<row,col,2>`, $\cdots$, `Q<row,col,9>`

is `true`.

In [17]:
function exactlyOneDigit(row: number, col: number): Set<Clause> {
    return exactlyOne(range(1, 9).map(d => varName(row, col, d)));
}

The function `constraintsFromPuzzle`  returns a set of clauses stating that the variables corresponding to numbers that are already given in the Sudoku puzzle take the values that are specified.

In [18]:
function constraintsFromPuzzle(): Set<Clause> {
    const Puzzle  = createPuzzle();
    const Indices = range(1, 9);    
    return Indices.cartesianProduct(Indices).filterMap(
        ([r, c]) => Puzzle[r-1][c-1] != '*',
        ([r, c]) => set(varName(r, c, Puzzle[r-1][c-1] as number)));
}

In [19]:
for (const clause of constraintsFromPuzzle()) {
    console.log(clause.toString()); 
}
constraintsFromPuzzle().size

{Q<1,1,8>}
{Q<2,3,3>}
{Q<2,4,6>}
{Q<3,2,7>}
{Q<3,5,9>}
{Q<3,7,2>}
{Q<4,2,5>}
{Q<4,6,7>}
{Q<5,5,4>}
{Q<5,6,5>}
{Q<5,7,7>}
{Q<6,4,1>}
{Q<6,8,3>}
{Q<7,3,1>}
{Q<7,8,6>}
{Q<7,9,8>}
{Q<8,3,8>}
{Q<8,4,5>}
{Q<8,8,1>}
{Q<9,2,9>}
{Q<9,7,4>}
21


The function `getBlockCells` calculates the absolute grid coordinates for all 9 cells 
within a specific 3x3 Sudoku block.  The Sudoku board is divided into a 3x3 grid of blocks. 
This function takes the zero-based indices of a block and maps its local cells to the 
1-based coordinates used by the solver variables. 
 * `r` - The row index of the 3x3 macro-block (must be 0, 1, or 2).
 * `c` - The column index of the 3x3 macro-block (must be 0, 1, or 2).
The function returns an array of 9 tuples, where each tuple `[row, col]` represents 
 the 1-based row and column coordinates on the full 9x9 Sudoku board.
 
For example, the set of coordinates for the top-middle block (`r = 0, c = 1`) 
is given as follows:
 ```getBlockCells(0, 1) = 
         [[1, 4], [1, 5], [1, 6], 
          [2, 4], [2, 5], [2, 6], 
          [3, 4], [3, 5], [3, 6]]
```

In [20]:
function getBlockCells(r: number, c: number): Set<Position> {
    return range(1, 3).cartesianProduct(range(1, 3)).map(
        ([row, col]) => tpl(r * 3 + row, c * 3 + col)
    );
}

In [21]:
getBlockCells(0, 1)

{(1, 4), (1, 5), (1, 6), (2, 4), (2, 5), (2, 6), (3, 4), (3, 5), (3, 6)}


The function `allConstraints` returns a CSP that encodes the given sudoku as a CSP.

In [22]:
function allConstraints(): Set<Clause> {
    // 1. Start with the constraints from the puzzle
    const Clauses = constraintsFromPuzzle();
    // 2. There is exactly one digit in every field
    Clauses.flatMap(
        range(1, 9).cartesianProduct(range(1, 9)), 
        ([row, col]) => exactlyOneDigit(row, col)
    );
    // 3. All entries in a row are unique
    Clauses.flatMap(
        range(1, 9), 
        row => exactlyOneForEveryDigit(range(1, 9).map(col => tpl(row, col)))
    );
    // 4. All entries in a column are unique
    Clauses.flatMap(
        range(1, 9), 
        col => exactlyOneForEveryDigit(range(1, 9).map(row => tpl(row, col)))
    );
    // 5. All entries in a 3x3 square are unique
    const block = range(0, 2).cartesianProduct(range(0, 2));
    Clauses.flatMap(block, ([r, c]) => exactlyOneForEveryDigit(getBlockCells(r, c)));
    return Clauses;
}

In [23]:
allConstraints().size

10551


In [24]:
function sudoku(): Set<Clause> | null {
    const Clauses  = allConstraints();
    const Solution = solve(Clauses);
    const EmptyClause = set<Literal>();
    if (!Solution.has(EmptyClause)) {
        return Solution;
    } else {
        console.log('The problem is not solvable!');
        return null;
    }
}

On my 2022 Mac Studio this takes 21 minutes.

In [25]:
console.time('sudoku');
const Solution = sudoku();
console.timeEnd('sudoku');

sudoku: 18:35.194 (m:ss.mmm)


In [26]:
Solution

{{(¬, Q<1,1,1>)}, {(¬, Q<1,1,2>)}, {(¬, Q<1,1,3>)}, {(¬, Q<1,1,4>)}, {(¬, Q<1,1,5>)}, {(¬, Q<1,1,6>)}, {(¬, Q<1,1,7>)}, {(¬, Q<1,1,9>)}, {(¬, Q<1,2,2>)}, {(¬, Q<1,2,3>)}, {(¬, Q<1,2,4>)}, {(¬, Q<1,2,5>)}, {(¬, Q<1,2,6>)}, {(¬, Q<1,2,7>)}, {(¬, Q<1,2,8>)}, {(¬, Q<1,2,9>)}, {(¬, Q<1,3,1>)}, {(¬, Q<1,3,3>)}, {(¬, Q<1,3,4>)}, {(¬, Q<1,3,5>)}, {(¬, Q<1,3,6>)}, {(¬, Q<1,3,7>)}, {(¬, Q<1,3,8>)}, {(¬, Q<1,3,9>)}, {(¬, Q<1,4,1>)}, {(¬, Q<1,4,2>)}, {(¬, Q<1,4,3>)}, {(¬, Q<1,4,4>)}, {(¬, Q<1,4,5>)}, {(¬, Q<1,4,6>)}, {(¬, Q<1,4,8>)}, {(¬, Q<1,4,9>)}, {(¬, Q<1,5,1>)}, {(¬, Q<1,5,2>)}, {(¬, Q<1,5,3>)}, {(¬, Q<1,5,4>)}, {(¬, Q<1,5,6>)}, {(¬, Q<1,5,7>)}, {(¬, Q<1,5,8>)}, {(¬, Q<1,5,9>)}, {(¬, Q<1,6,1>)}, {(¬, Q<1,6,2>)}, {(¬, Q<1,6,4>)}, {(¬, Q<1,6,5>)}, {(¬, Q<1,6,6>)}, {(¬, Q<1,6,7>)}, {(¬, Q<1,6,8>)}, {(¬, Q<1,6,9>)}, {(¬, Q<1,7,1>)}, {(¬, Q<1,7,2>)}, {(¬, Q<1,7,3>)}, {(¬, Q<1,7,4>)}, {(¬, Q<1,7,5>)}, {(¬, Q<1,7,7>)}, {(¬, Q<1,7,8>)}, {(¬, Q<1,7,9>)}, {(¬, Q<1,8,1>)}, {(¬, Q<1,8,2>)}, {(¬, Q<1,8,3>

## Graphical Representation

In [27]:
function arb<T extends Value>(S: Set<T>): T {
    return S.pickRandom();
}

In [28]:
function transformSolution(Solution: Set<Clause>): Record<string, number> {
    const Result: Record<string, number> = {};
    for (const UnitClause of Solution) {
        const literal = arb(UnitClause); 
        if (literal != null && typeof literal == 'string') {
            const matches = literal.match(/(\d+),(\d+),(\d+)/);
            if (matches) {
                const row = parseInt(matches[1], 10);
                const col = parseInt(matches[2], 10);
                const digit = parseInt(matches[3], 10);
                Result[`V${row}${col}`] = digit;
            }
        }
    }
    return Result;
}

In [29]:
import { display } from 'tslab';

function showSolution(Solution: Set<Clause>, width: string = '50%'): void {
    const solutionMap: Record<string, number> = transformSolution(Solution);
    const Sudoku = createPuzzle();
    for (let row = 0; row < 9; row++) {
        for (let col = 0; col < 9; col++) {
            if (Sudoku[row][col] != '*') {
                delete solutionMap[`V${row + 1}${col + 1}`];
            }
        }
    }
    let html = `<table style="width:${width}; border-collapse: collapse; border: 2px solid black; font-family: sans-serif;">`;
    for (let row = 0; row < 9; row++) {
        html += '<tr>';
        for (let col = 0; col < 9; col++) {
            const key = `V${row + 1}${col + 1}`;
            let value: number | string | undefined = solutionMap[key];
            const original = Sudoku[row][col];
            let cellStyle = "font-weight: normal;";
            if (original != '*') {
                value = original;
                cellStyle = "font-weight: bold;";
            }
            const blockRow = Math.floor(row / 3);
            const blockCol = Math.floor(col / 3);
            const isGray = (blockRow + blockCol) % 2 != 0;
            const bgColor = isGray ? '#f0f0f0' : '#ffffff';
            let borderStyle = "border: 1px solid #ccc;";
            if ((col + 1) % 3 == 0 && col < 8) {
                borderStyle += "border-right: 2px solid black;";
            }
            if ((row + 1) % 3 == 0 && row < 8) {
                borderStyle += "border-bottom: 2px solid black;";
            }
            html += `<td style="${borderStyle} width:30px; height:30px; text-align:center; font-size:20px; background-color:${bgColor}; ${cellStyle}">${value !== undefined ? value : ''}</td>`;
        }
        html += '</tr>';
    }
    html += '</table>';
    display.html(html);
}

In [30]:
showSolution(Solution);

8,1,2,7,5,3,6,4,9
9,4,3,6,8,2,1,7,5
6,7,5,4,9,1,2,8,3
1,5,4,2,3,7,8,9,6
3,6,9,8,4,5,7,2,1
2,8,7,1,6,9,5,3,4
5,2,1,9,7,4,3,6,8
4,3,8,5,2,6,9,1,7
7,9,6,3,1,8,4,5,2
